# B04a — Decision Trees and XGBoost

**COMPSCI 713 — Week 4: Expert Systems & Tree-Based Learning**

---

## Overview

Decision trees are the bridge between symbolic AI (rule-based expert systems) and modern machine learning. The ID3 algorithm, invented in 1986, showed that machines could *induce* rules from data — a key step beyond hand-crafted expert systems like MYCIN.

In this lesson you will:
- Understand how decision trees make decisions
- Implement the ID3 algorithm using information gain
- Build and visualise decision trees with scikit-learn
- Understand overfitting and pruning
- Learn ensemble methods: bagging (Random Forest) and boosting (XGBoost)
- Connect decision trees to expert systems and rule extraction

### COMPSCI 713 Alignment
- **Week 4 Thursday:** ID3 to XGBoost (decision trees, tree induction, ensembles)

## 1. What Is a Decision Tree?

A decision tree is a flowchart-like structure where:
- Each **internal node** tests a feature (e.g., "Is temperature > 25°C?")
- Each **branch** represents an outcome of the test
- Each **leaf node** gives a prediction (class label or value)

```
         [Outlook?]
        /     |     \
    Sunny   Overcast  Rain
      |        |        |
  [Humidity?] YES   [Windy?]
   /      \          /    \
 High    Normal   Strong  Weak
  NO      YES      NO     YES
```

This is the classic **Play Tennis** dataset — one of the first examples used to demonstrate ID3.

### Connection to Expert Systems
A decision tree can be read as a set of IF-THEN rules:
- IF Outlook=Sunny AND Humidity=Normal → Play Tennis
- IF Outlook=Overcast → Play Tennis
- IF Outlook=Rain AND Windy=Weak → Play Tennis

The key difference from MYCIN: these rules are **learned from data**, not hand-crafted by experts.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install scikit-learn xgboost matplotlib pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import math

# Classic Play Tennis dataset
data = {
    'Outlook':    ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast',
                   'Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature':['Hot','Hot','Hot','Mild','Cool','Cool','Cool',
                   'Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity':   ['High','High','High','High','Normal','Normal','Normal',
                   'High','Normal','Normal','Normal','High','Normal','High'],
    'Windy':      [False,True,False,False,False,True,True,
                   False,False,False,True,True,False,True],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes',
                   'No','Yes','Yes','Yes','Yes','Yes','No']
}

df = pd.DataFrame(data)
print(df)
print(f"\nClass distribution: {Counter(df['PlayTennis'])}")

## 2. Information Gain — The Heart of ID3

ID3 (Iterative Dichotomiser 3, Quinlan 1986) chooses the **best feature to split on** at each node using **Information Gain**.

### Entropy
Entropy measures the **impurity** (uncertainty) of a set:

$$H(S) = -\sum_{c} p_c \log_2 p_c$$

- H = 0: perfectly pure (all same class)
- H = 1: maximally impure (50/50 split for binary)

### Information Gain
$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

We pick the feature A that **maximises** IG — it gives us the most information about the class.

In [ ]:
def entropy(labels):
    """Calculate Shannon entropy of a label list."""
    n = len(labels)
    if n == 0:
        return 0
    counts = Counter(labels)
    return -sum((c/n) * math.log2(c/n) for c in counts.values() if c > 0)

def information_gain(df, feature, target='PlayTennis'):
    """Calculate information gain of splitting on a feature."""
    total_entropy = entropy(df[target])
    n = len(df)
    # Weighted entropy after split
    weighted = sum(
        (len(subset) / n) * entropy(subset[target])
        for _, subset in df.groupby(feature)
    )
    return total_entropy - weighted

# Calculate entropy of the full dataset
H = entropy(df['PlayTennis'])
print(f"Dataset entropy H(S) = {H:.4f}")
print()

# Information gain for each feature
features = ['Outlook', 'Temperature', 'Humidity', 'Windy']
gains = {f: information_gain(df, f) for f in features}

print("Information Gain per feature:")
for f, g in sorted(gains.items(), key=lambda x: -x[1]):
    print(f"  IG({f:12s}) = {g:.4f}")

best = max(gains, key=gains.get)
print(f"\n→ ID3 splits on: {best} (highest IG)")

## 3. Building a Decision Tree with scikit-learn

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder

# Encode categorical features
df_enc = df.copy()
le = LabelEncoder()
for col in ['Outlook', 'Temperature', 'Humidity', 'PlayTennis']:
    df_enc[col] = le.fit_transform(df_enc[col])
df_enc['Windy'] = df_enc['Windy'].astype(int)

X = df_enc[features]
y = df_enc['PlayTennis']

# Train a decision tree (criterion='entropy' = ID3 style)
tree = DecisionTreeClassifier(criterion='entropy', random_state=42)
tree.fit(X, y)

# Visualise
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(tree, feature_names=features, class_names=['No','Yes'],
          filled=True, rounded=True, ax=ax)
plt.title('Decision Tree (ID3 / Entropy criterion)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Tree depth: {tree.get_depth()}")
print(f"Number of leaves: {tree.get_n_leaves()}")

## 4. Overfitting and Pruning

A fully grown decision tree **memorises** the training data — it overfits.

**Pruning** strategies:
- **Pre-pruning:** Stop growing early (`max_depth`, `min_samples_leaf`)
- **Post-pruning:** Grow full tree, then remove branches that don't improve validation accuracy
- **Cost-complexity pruning (ccp_alpha):** scikit-learn's built-in post-pruning

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Use Iris for a clearer overfitting demo
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

depths = range(1, 15)
train_accs, test_accs = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, t.predict(X_train)))
    test_accs.append(accuracy_score(y_test, t.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(depths, train_accs, 'b-o', label='Train accuracy')
plt.plot(depths, test_accs, 'r-o', label='Test accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Overfitting vs Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[test_accs.index(max(test_accs))]
print(f"Best test accuracy at depth={best_depth}: {max(test_accs):.3f}")

## 5. Ensemble Methods

Single decision trees are unstable — small changes in data produce very different trees. **Ensembles** combine many trees to get a more robust prediction.

### Bagging — Random Forest
- Train N trees on **random subsets** of data (bootstrap sampling)
- Each tree also uses a **random subset of features**
- Final prediction = **majority vote** (classification) or **average** (regression)
- Reduces **variance** (overfitting)

### Boosting — XGBoost
- Train trees **sequentially**: each tree corrects the errors of the previous
- Uses **gradient descent** in function space
- Reduces **bias** (underfitting) and variance
- XGBoost (Chen & Guestrin, 2016) adds regularisation, parallel processing, and handles missing values

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import make_classification

# Generate a harder classification problem
X_c, y_c = make_classification(n_samples=1000, n_features=20, n_informative=10,
                                 n_redundant=5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

models = {
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest (100 trees)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':<35} {'Train Acc':>10} {'Test Acc':>10}")
print('-' * 57)
for name, model in models.items():
    model.fit(X_tr, y_tr)
    tr_acc = accuracy_score(y_tr, model.predict(X_tr))
    te_acc = accuracy_score(y_te, model.predict(X_te))
    print(f"{name:<35} {tr_acc:>10.3f} {te_acc:>10.3f}")

## 6. XGBoost — Extreme Gradient Boosting

In [ ]:
try:
    import xgboost as xgb

    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_model.fit(X_tr, y_tr)
    xgb_acc = accuracy_score(y_te, xgb_model.predict(X_te))
    print(f"XGBoost test accuracy: {xgb_acc:.3f}")

    # Feature importance
    importances = xgb_model.feature_importances_
    top_idx = np.argsort(importances)[-10:]
    plt.figure(figsize=(8, 4))
    plt.barh(range(10), importances[top_idx], color='steelblue')
    plt.yticks(range(10), [f'Feature {i}' for i in top_idx])
    plt.xlabel('Importance')
    plt.title('XGBoost Feature Importance (Top 10)')
    plt.tight_layout()
    plt.show()

except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    print("Using sklearn GradientBoosting as equivalent demo.")

## 7. Extracting Rules from a Decision Tree

One of the key advantages of decision trees over neural networks: **interpretability**. We can extract human-readable IF-THEN rules — just like an expert system, but learned from data.

In [ ]:
from sklearn.tree import export_text

# Train a shallow tree on Iris for readable rules
iris_tree = DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=42)
iris_tree.fit(X_iris, y_iris)

rules = export_text(iris_tree, feature_names=list(iris.feature_names))
print("Extracted IF-THEN rules from Decision Tree:")
print(rules)

## 8. Summary

| Algorithm | Key Idea | Strength | Weakness |
|---|---|---|---|
| ID3 | Maximise information gain | Interpretable, fast | Overfits, only categorical |
| C4.5 | Gain ratio + pruning | Handles continuous features | Slower |
| CART | Gini impurity | Regression + classification | Greedy splits |
| Random Forest | Bagging + random features | Low variance, robust | Less interpretable |
| XGBoost | Sequential boosting + regularisation | State-of-the-art on tabular data | Many hyperparameters |

### Key Takeaways
- Decision trees **induce rules from data** — the machine learning answer to hand-crafted expert systems
- **Information gain** (entropy reduction) guides the ID3 splitting criterion
- Trees overfit easily — **pruning** and **ensembles** are the solution
- **Random Forest** reduces variance via bagging; **XGBoost** reduces bias via boosting
- Trees remain highly competitive on **tabular/structured data** even against deep learning

### Next Steps
- **B05** — Neural Network Fundamentals (the other side of the coin)
- **I05** — Transfer Learning (combining learned representations)
- **I14** — Explainable AI (SHAP values for XGBoost)